# Semana 02: Sensoriamento Discreto, Analógico, PWM e Condicionamento de Sinal para TI

## Módulo de Aquisição de Dados — Fábrica Virtual Smart N1

Este notebook aborda o funcionamento de sensores discretos (indutivos, capacitivos, fim de curso), sensores analógicos e modulação PWM **sob a perspectiva da Engenharia de Software e TI**, detalhando a interpretação lógica (Active-High/Low, Fail-Safe), resolução de ADC e filtragem de ruído digital (*Debounce*).

### Objetivos de aprendizagem
- Interpretar sinais discretos sob a perspectiva de software: Lógica NA vs NF (Fail-Safe), PNP (Active-High) vs NPN (Active-Low).
- Compreender a amostragem de sinais contínuos (0-10V / 4-20mA) e o conceito de *Live Zero* (4mA) para diagnóstico de falhas na aplicação.
- Analisar a geração e interpretação de sinais PWM (Frequência, Duty Cycle, Tensão Média Equivalente $V_{avg}$).
- Entender o efeito de ricochete de contato (*Contact Bounce*) e implementar algoritmos de filtro digital (*Software Debounce*).
- Executar scripts Python para cálculo de resolução ADC, decodificação PWM e amostragem com filtro de ruído.

---


## 1. Fundamentação teórica

### 1.1 Interpretação Lógica de Sensores Discretos em Software

![Esquema Lógico PNP e NPN e Filtro Debounce](img/pnp_npn_logic_ti.jpg)

Para a aplicação de software, o acionamento de um sensor discreto se traduz na alteração de um bit de memória no controlador. No entanto, a forma como o circuito elétrico é construído altera a interpretação no código:

#### 1. Lógica de Contato: NA (Normally Open) vs NF (Normally Closed)
- **Contato NA (Normally Open):** Sem acionamento = Bit `0` (Falso). Quando acionado = Bit `1` (Verdadeiro).
- **Contato NF (Normally Closed — Fail-Safe):** Sem acionamento = Bit `1` (Verdadeiro). Quando acionado ou **em caso de rompimento de cabo** = Bit `0` (Falso). **Uso obrigatório em botões de emergência**, pois a interrupção da fiação é interpretada pelo software imediatamente como falha/parada de segurança.

#### 2. Transistores de Saída: PNP (Active-High) vs NPN (Active-Low)
- **Sensor PNP (Sourcing / Active-High):** Ao ser acionado, injeta $+24\text{V DC}$ na entrada do controlador, registrando Nível Lógico ALTO (`True`). Padrão predominante na Europa e Brasil.
- **Sensor NPN (Sinking / Active-Low):** Ao ser acionado, conecta a entrada ao $0\text{V GND}$, registrando Nível Lógico BAIXO (`False`). O software deve inverter a lógica (`state = NOT input`).


### 1.2 Sinais Contínuos e a Técnica do "Live Zero" (4-20mA)

Sinais analógicos representam grandezas variáveis em uma faixa contínua:

- **Sinal de Tensão ($0\text{ a }10\text{V}$):** Suscetível a quedas de tensão ao longo de cabos longos e ruídos eletromagnéticos.
- **Sinal de Corrente ($4\text{ a }20\text{mA}$):** Imune a quedas de tensão em cabos longos. Utiliza o padrão **Live Zero** ($4\text{mA}$):
  - $4\text{mA} = 0\%$ da escala da grandeza (ex: $0\text{ bar}$).
  - $20\text{mA} = 100\%$ da escala da grandeza (ex: $10\text{ bar}$).
  - **Se o sinal for $0\text{mA}$:** O software de TI identifica **imediatamente um erro de cabo partido / sensor desconectado**, em vez de confundir com leitura zero!


### 1.3 Sinais PWM (Pulse Width Modulation) e Controle de Atuadores

O sinal **PWM** é uma forma eficiente de controlar a potência entregue a motores, elementos de aquecimento e atuadores utilizando saídas digitais em alta velocidade.

$$\text{Duty Cycle } (D) = \frac{t_{on}}{t_{on} + t_{off}} \times 100\%$$

$$\text{Tensão Média Equivalent } (V_{avg}) = D \times V_{max}$$

| Duty Cycle ($D$) | $t_{on}$ vs $T$ | Tensão Média ($V_{max} = 24\text{V}$) | Aplicação no Software |
| :---: | :---: | :---: | :--- |
| **0%** | $0\text{ ms}$ ativo | $0,0\text{ V}$ | Atuador Desligado / Parado |
| **25%** | $1/4$ do período | $6,0\text{ V}$ | Velocidade Baixa ($25\%$ de rotação) |
| **50%** | Metade do período | $12,0\text{ V}$ | Velocidade Média ($50\%$ de rotação) |
| **75%** | $3/4$ do período | $18,0\text{ V}$ | Velocidade Alta ($75\%$ de rotação) |
| **100%** | Período completo | $24,0\text{ V}$ | Potência Máxima / Carga Total |

---


## 2. Arquitetura da atividade

**Sinal Elétrico de Campo → ADC 12-bits / Leitor PWM → Algoritmo de Debounce (Filtro 5ms) → Registro de Memória de TI**

---


## 3. Prática — Amostragem de Sinais, Decodificação PWM e Debounce em Python

### Passo 1 — Diagnóstico de Cabo Rompido em Sinal 4-20mA (Live Zero)
Execute o código em Python para testar a detecção de erro em malhas de corrente 4-20mA.

In [ ]:
def interpretar_sinal_4_20ma(corrente_ma, pressao_min_bar=0.0, pressao_max_bar=10.0):
    if corrente_ma < 3.5:
        return {"status": "ERRO_FALHA_CABO_ROMPIDO", "pressao_bar": None}
    elif corrente_ma > 20.5:
        return {"status": "ERRO_SOBRECARGA_CORRENTE", "pressao_bar": None}
    else:
        # Mapeamento linear de 4mA a 20mA
        percentual = (corrente_ma - 4.0) / (20.0 - 4.0)
        pressao = pressao_min_bar + (percentual * (pressao_max_bar - pressao_min_bar))
        return {"status": "OPERACAO_NORMAL", "pressao_bar": round(pressao, 2)}

# Teste com leituras normais e falhas
leituras = [4.0, 12.0, 20.0, 0.0] # 0mA simula cabo cortado
print("=== DIAGNÓSTICO DE SINAL 4-20mA (LIVE ZERO) ===\n")
for c in leituras:
    res = interpretar_sinal_4_20ma(c)
    print(f"Corrente Medida: {c:4.1f} mA | Status: {res['status']:<25} | Pressão: {res['pressao_bar']}")


### Passo 2 — Filtro Digital de Debounce em Software
Execute o algoritmo de debounce para tratar sinais discretos com ricochete mecânico de contato.

In [ ]:
def aplicar_filtro_debounce(sinal_bruto, janela_estabilidade=5):
    sinal_limpo = []
    estado_estavel = 0
    contador = 0
    
    for amostra in sinal_bruto:
        if amostra != estado_estavel:
            contador += 1
            if contador >= janela_estabilidade:
                estado_estavel = amostra
                contador = 0
        else:
            contador = 0
        sinal_limpo.append(estado_estavel)
    return sinal_limpo

# Amostras brutas de um botão mecânico com ruído inicial
sinal_ruidoso = [0, 0, 1, 0, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1]
sinal_filtrado = aplicar_filtro_debounce(sinal_ruidoso, janela_estabilidade=4)

print("Sinal Bruto do Sensor: ", sinal_ruidoso)
print("Sinal Limpo em Software:", sinal_filtrado)


---

## 4. Exercícios de fixação e avaliação

### Questão 1
Por que a técnica do *Live Zero* ($4\text{ a }20\text{mA}$) é amplamente utilizada em sensores analógicos industriais e de que forma o software diferencia uma leitura de zero real de um cabo rompido?

### Questão 2
Em um sistema com sensores discretos PNP (Active-High), qual o nível lógico registrado na memória do controlador quando o sensor é ativado? O que muda se o sensor for NPN (Active-Low)?

### Questão 3
Uma saída PWM opera a $24\text{V}$ alimentando uma resistência de aquecimento industrial. Se o software definir o Duty Cycle em $65\%$, qual será a tensão média equivalente ($V_{avg}$) entregue ao aquecedor?
